In [ ]:
from write_features_functions import *
from util import *
import os
import json
import geojson
import h5py
import numpy as np

In [ ]:
# Define paths and parameters
pth0 = r'\\10.162.80.16\Andre_expansion\data\Skin Lymphedema'
nm = 'L0003_diseased'
dtm = '9_17_2024'
model = 'pdac'

pthstardist = os.path.join(pth0, nm, f'StarDist_{dtm}_{model}')
json_path = os.path.join(pthstardist, 'json')
selection_path = os.path.join(pthstardist, 'immune_cell_annotations_AF')
selection_json_path = os.path.join(selection_path, 'json')

os.makedirs(selection_json_path, exist_ok=True)

pthpkl = os.path.join(selection_path, 'stardist_feature_df_pickles')
os.makedirs(pthpkl, exist_ok=True)

pthpklmat = os.path.join(pthpkl, 'mat')
os.makedirs(pthpklmat, exist_ok=True)

pthidxs = os.path.join(selection_path, 'immune_idxs')
pthidxs_geojson = os.path.join(pthidxs, 'geojsons')
os.makedirs(pthidxs_geojson, exist_ok=True)

In [ ]:
# Function to get sorted files
def get_sorted_files(directory, *extensions, filter_str=None):
    return sorted([
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if f.endswith(extensions) and (filter_str in f if filter_str else True)
    ])

# Function to downsample data
def get_ds_data(segmentation_data, ds):
    data_list = []
    for data in segmentation_data:
        centroid = data['centroid'][0]
        contour = data['contour'][0]

        ds_centroid = [int(c / ds) for c in centroid]
        ds_contour = [[value / ds for value in sublist] for sublist in contour]
        ds_contour = [[round(x, 2), round(y, 2)] for x, y in zip(ds_contour[0], ds_contour[1])]

        data_list.append([ds_centroid, ds_contour])
    return data_list

In [ ]:
# Get list of JSON files
json_pth_list = get_sorted_files(json_path, '.json')
# Generate GeoJSON for immune cell annotations
ds_amt = 1  # 20x magnification

for p, file in enumerate(json_pth_list):
    imnm = os.path.splitext(os.path.basename(file))[0]
    print(f'{imnm}  {p + 1}/{len(json_pth_list)}')

    geojson_filepth = os.path.join(pthidxs_geojson, f'{imnm}.geojson')
    mat_filepth = os.path.join(pthidxs, f'{imnm}.mat')

    if not os.path.exists(geojson_filepth):
        with open(file) as f:
            segmentation_data = json.load(f)

        with h5py.File(mat_filepth, 'r') as mat_data:
            idx_list = np.array(mat_data['cluster1']).flatten()

        data_list = get_ds_data(segmentation_data, ds_amt)
        GEOdata = []

        for j, (centroid, contour) in enumerate(data_list):
            centroid = [centroid[0], centroid[1]]
            contour = [[coord for coord in xy[::-1]] for xy in contour]
            contour.append(contour[0])  # Close the contour

            if idx_list[j] == 1:
                classification_name = 'immune'
                color = [255, 165, 0]  # Light orange
            else:
                classification_name = 'others'
                color = [97, 214, 59]  # Green

            dict_data = {
                "type": "Feature",
                "id": "PathCellObject",
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [contour]
                },
                "properties": {
                    'objectType': 'annotation',
                    'classification': {'name': classification_name, 'color': color}
                }
            }

            GEOdata.append(dict_data)

        with open(geojson_filepth, 'w') as outfile:
            geojson.dump(GEOdata, outfile)
        print('Finished', imnm)

    else:
        print(f'Skipping {imnm}')